# Module 05: Generative Adversarial Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/05-gans/notebook.ipynb)

**GPU recommended:** Yes (DCGAN training is ~5 min on GPU, ~20 min on CPU).

## Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

plt.style.use("seaborn-v0_8-whitegrid")

---
## Part 1: Simple GAN on 2D Gaussian Mixture

We start with a toy problem: learning a mixture of 2D Gaussians.
This makes it easy to visualize how the generator distribution evolves during training.

### 1.1 Target Distribution

Our target is a mixture of 8 Gaussians arranged in a ring.

In [ ]:
def sample_ring_mixture(n_samples, n_components=8, radius=2.0, std=0.05):
    """Sample from a ring of 2D Gaussians."""
    angles = np.linspace(0, 2 * np.pi, n_components, endpoint=False)
    centers = np.stack([radius * np.cos(angles), radius * np.sin(angles)], axis=1)
    indices = np.random.randint(0, n_components, size=n_samples)
    samples = centers[indices] + np.random.randn(n_samples, 2) * std
    return torch.tensor(samples, dtype=torch.float32)


real_samples = sample_ring_mixture(2000)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(real_samples[:, 0], real_samples[:, 1], s=4, alpha=0.5)
ax.set_title("Target: 8-component Gaussian ring")
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### 1.2 Generator and Discriminator MLPs

In [ ]:
class SimpleGenerator(nn.Module):
    def __init__(self, latent_dim=2, hidden_dim=128, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class SimpleDiscriminator(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

### 1.3 Train the Simple GAN and Visualize Evolution

In [ ]:
latent_dim = 2
gen_2d = SimpleGenerator(latent_dim=latent_dim).to(device)
disc_2d = SimpleDiscriminator().to(device)

opt_g = optim.Adam(gen_2d.parameters(), lr=1e-3, betas=(0.5, 0.999))
opt_d = optim.Adam(disc_2d.parameters(), lr=1e-3, betas=(0.5, 0.999))
criterion = nn.BCELoss()

n_steps = 5000
batch_size = 256
snapshot_steps = [0, 500, 1000, 2000, 5000]
snapshots = {}

for step in range(n_steps + 1):
    real = sample_ring_mixture(batch_size).to(device)
    z = torch.randn(batch_size, latent_dim, device=device)
    fake = gen_2d(z)

    # --- Discriminator step ---
    d_real = disc_2d(real)
    d_fake = disc_2d(fake.detach())
    loss_d = criterion(d_real, torch.ones_like(d_real)) + criterion(
        d_fake, torch.zeros_like(d_fake)
    )
    opt_d.zero_grad()
    loss_d.backward()
    opt_d.step()

    # --- Generator step ---
    d_fake = disc_2d(fake)
    loss_g = criterion(d_fake, torch.ones_like(d_fake))
    opt_g.zero_grad()
    loss_g.backward()
    opt_g.step()

    if step in snapshot_steps:
        with torch.no_grad():
            z_vis = torch.randn(2000, latent_dim, device=device)
            snapshots[step] = gen_2d(z_vis).cpu().numpy()

print("Training complete.")

In [ ]:
fig, axes = plt.subplots(1, len(snapshot_steps), figsize=(4 * len(snapshot_steps), 4))
for ax, step in zip(axes, snapshot_steps):
    pts = snapshots[step]
    ax.scatter(real_samples[:, 0], real_samples[:, 1], s=4, alpha=0.3, label="Real")
    ax.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.3, label="Generated")
    ax.set_title(f"Step {step}")
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect("equal")
    ax.legend(markerscale=4, fontsize=8)
plt.suptitle("Generator distribution evolution", fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 2: DCGAN on MNIST

We now move to image generation using Deep Convolutional GAN (DCGAN).
Key architectural guidelines from Radford et al. (2015):
- Replace pooling with strided convolutions (discriminator) and transposed convolutions (generator)
- Use batch normalization in both networks (except output layer of G, input layer of D)
- Use ReLU in generator, LeakyReLU in discriminator

### 2.1 MNIST DataLoader

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)

mnist_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
dataloader = DataLoader(mnist_dataset, batch_size=128, shuffle=True, num_workers=2)

### 2.2 DCGAN Generator and Discriminator

In [ ]:
nz = 100  # latent vector size
ngf = 64  # generator feature map size
ndf = 64  # discriminator feature map size
nc = 1  # number of channels (grayscale)


class DCGenerator(nn.Module):
    """DCGAN generator: maps latent vector z -> 32x32 image."""

    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            # z (nz x 1 x 1) -> (ngf*4 x 4 x 4)
            nn.ConvTranspose2d(nz, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # -> (ngf*2 x 8 x 8)
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # -> (ngf x 16 x 16)
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # -> (nc x 32 x 32)
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.main(z)


class DCDiscriminator(nn.Module):
    """DCGAN discriminator: maps 32x32 image -> real/fake score."""

    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            # (nc x 32 x 32) -> (ndf x 16 x 16)
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (ndf*2 x 8 x 8)
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (ndf*4 x 4 x 4)
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (1 x 1 x 1)
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)


def weights_init(m):
    """Custom weight initialization as per DCGAN paper."""
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

### 2.3 Train DCGAN

In [ ]:
torch.manual_seed(42)

netG = DCGenerator().to(device)
netD = DCDiscriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=2e-4, betas=(0.5, 0.999))

fixed_noise = torch.randn(64, nz, 1, 1, device=device)

num_epochs = 10
G_losses = []
D_losses = []
epoch_samples = {}

print("Starting DCGAN training...")
for epoch in range(num_epochs):
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        b_size = real_imgs.size(0)

        # --- Train Discriminator ---
        netD.zero_grad()
        label_real = torch.ones(b_size, device=device)
        label_fake = torch.zeros(b_size, device=device)

        output_real = netD(real_imgs)
        loss_d_real = criterion(output_real, label_real)
        loss_d_real.backward()

        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake_imgs = netG(noise)
        output_fake = netD(fake_imgs.detach())
        loss_d_fake = criterion(output_fake, label_fake)
        loss_d_fake.backward()

        loss_d = loss_d_real + loss_d_fake
        optimizerD.step()

        # --- Train Generator ---
        netG.zero_grad()
        output_fake = netD(fake_imgs)
        loss_g = criterion(output_fake, label_real)
        loss_g.backward()
        optimizerG.step()

        G_losses.append(loss_g.item())
        D_losses.append(loss_d.item())

    # Save generated samples at the end of each epoch
    with torch.no_grad():
        epoch_samples[epoch] = netG(fixed_noise).cpu()

    print(
        f"Epoch [{epoch+1}/{num_epochs}]  "
        f"D_loss: {loss_d.item():.4f}  G_loss: {loss_g.item():.4f}"
    )

print("DCGAN training complete.")

---
## Part 3: Training Dynamics

Monitoring discriminator and generator losses over time is crucial for diagnosing
training problems. Healthy GAN training typically shows both losses oscillating
without one dominating the other.

### 3.1 Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(G_losses, label="Generator", alpha=0.7, linewidth=0.5)
ax.plot(D_losses, label="Discriminator", alpha=0.7, linewidth=0.5)
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.set_title("DCGAN Training Losses")
ax.legend()
plt.tight_layout()
plt.show()

### 3.2 Smoothed Losses and Mode Collapse Detection

Mode collapse can be detected when generator loss drops very low while discriminator
loss stays high, or when generated sample diversity suddenly decreases. Here we
use a rolling average for a clearer view and compute a simple diversity metric.

In [ ]:
def rolling_average(values, window=100):
    """Compute a rolling average over a list of values."""
    cumsum = np.cumsum(np.insert(values, 0, 0))
    return (cumsum[window:] - cumsum[:-window]) / float(window)


fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Smoothed losses
window = 50
axes[0].plot(rolling_average(G_losses, window), label="Generator (smoothed)", linewidth=1.2)
axes[0].plot(rolling_average(D_losses, window), label="Discriminator (smoothed)", linewidth=1.2)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Loss")
axes[0].set_title("Smoothed Training Losses")
axes[0].legend()

# Loss ratio as a simple instability indicator
g_smooth = rolling_average(G_losses, window)
d_smooth = rolling_average(D_losses, window)
min_len = min(len(g_smooth), len(d_smooth))
ratio = g_smooth[:min_len] / (d_smooth[:min_len] + 1e-8)
axes[1].plot(ratio, color="red", linewidth=1.0)
axes[1].axhline(y=1.0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("G_loss / D_loss")
axes[1].set_title("Loss Ratio (mode collapse indicator)")

plt.tight_layout()
plt.show()

---
## Part 4: Generated Digit Grids at Different Epochs

In [ ]:
display_epochs = [0, 2, 4, 6, 9]

fig, axes = plt.subplots(1, len(display_epochs), figsize=(3 * len(display_epochs), 3))
for ax, ep in zip(axes, display_epochs):
    grid = vutils.make_grid(epoch_samples[ep], nrow=8, normalize=True, padding=1)
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap="gray")
    ax.set_title(f"Epoch {ep + 1}")
    ax.axis("off")
plt.suptitle("Generated MNIST digits over training", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# High-resolution grid from the final epoch
fig, ax = plt.subplots(figsize=(8, 8))
grid = vutils.make_grid(epoch_samples[num_epochs - 1], nrow=8, normalize=True, padding=2)
ax.imshow(grid.permute(1, 2, 0).numpy(), cmap="gray")
ax.set_title("Final epoch: 64 generated digits")
ax.axis("off")
plt.tight_layout()
plt.show()

---
## Part 5: Latent Space Interpolation

A well-trained generator should produce smooth transitions between two
points in latent space, demonstrating that it has learned a meaningful
continuous representation rather than memorizing discrete modes.

In [ ]:
def interpolate_latent(z1, z2, n_steps=10):
    """Linearly interpolate between two latent vectors."""
    alphas = torch.linspace(0, 1, n_steps).to(z1.device)
    interpolated = torch.stack([z1 * (1 - a) + z2 * a for a in alphas])
    return interpolated


def slerp(z1, z2, n_steps=10):
    """Spherical linear interpolation between two latent vectors."""
    z1_flat = z1.view(-1)
    z2_flat = z2.view(-1)
    omega = torch.acos(
        torch.clamp(
            torch.dot(z1_flat, z2_flat)
            / (torch.norm(z1_flat) * torch.norm(z2_flat)),
            -1.0,
            1.0,
        )
    )
    alphas = torch.linspace(0, 1, n_steps).to(z1.device)
    results = []
    for a in alphas:
        if omega.abs() < 1e-6:
            results.append(z1 * (1 - a) + z2 * a)
        else:
            results.append(
                (torch.sin((1 - a) * omega) / torch.sin(omega)) * z1
                + (torch.sin(a * omega) / torch.sin(omega)) * z2
            )
    return torch.stack(results)

In [ ]:
torch.manual_seed(42)
n_interpolations = 4
n_steps = 10

fig, axes = plt.subplots(n_interpolations, 1, figsize=(n_steps * 1.2, n_interpolations * 1.4))

netG.eval()
with torch.no_grad():
    for row in range(n_interpolations):
        z1 = torch.randn(1, nz, 1, 1, device=device)
        z2 = torch.randn(1, nz, 1, 1, device=device)
        z_interp = slerp(z1.squeeze(), z2.squeeze(), n_steps)
        z_interp = z_interp.view(n_steps, nz, 1, 1)
        imgs = netG(z_interp)
        grid = vutils.make_grid(imgs, nrow=n_steps, normalize=True, padding=1)
        axes[row].imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap="gray")
        axes[row].axis("off")
        axes[row].set_ylabel(f"Row {row+1}", fontsize=10, rotation=0, labelpad=35)

netG.train()
plt.suptitle("Latent space interpolation (spherical)", fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 6: WGAN-GP (Wasserstein GAN with Gradient Penalty)

The Wasserstein GAN replaces the BCE loss with the Wasserstein distance
(Earth Mover's distance). Instead of clipping weights, WGAN-GP enforces the
Lipschitz constraint via a gradient penalty term.

Key differences from vanilla GAN:
- No sigmoid in discriminator (now called "critic")
- Critic is trained more steps per generator step (typically 5:1)
- No log in the loss; maximize `E[D(x)] - E[D(G(z))]`
- Gradient penalty instead of weight clipping

### 6.1 WGAN-GP Critic (no sigmoid, no batch norm)

In [ ]:
class WGANCritic(nn.Module):
    """Critic network for WGAN-GP. Uses LayerNorm instead of BatchNorm."""

    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            # (nc x 32 x 32) -> (ndf x 16 x 16)
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (ndf*2 x 8 x 8)
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.LayerNorm([ndf * 2, 8, 8]),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (ndf*4 x 4 x 4)
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.LayerNorm([ndf * 4, 4, 4]),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (1 x 1 x 1)
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x):
        return self.main(x).view(-1)


class WGANGenerator(nn.Module):
    """Generator for WGAN-GP (same architecture as DCGAN generator)."""

    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.main(z)

### 6.2 Gradient Penalty

In [ ]:
def compute_gradient_penalty(critic, real_data, fake_data, device):
    """Compute gradient penalty for WGAN-GP."""
    batch_size = real_data.size(0)
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolated = (alpha * real_data + (1 - alpha) * fake_data).requires_grad_(True)

    critic_interp = critic(interpolated)

    gradients = torch.autograd.grad(
        outputs=critic_interp,
        inputs=interpolated,
        grad_outputs=torch.ones_like(critic_interp),
        create_graph=True,
        retain_graph=True,
    )[0]

    gradients = gradients.view(batch_size, -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

### 6.3 Train WGAN-GP

In [ ]:
torch.manual_seed(42)

wgan_G = WGANGenerator().to(device)
wgan_C = WGANCritic().to(device)
wgan_G.apply(weights_init)
wgan_C.apply(weights_init)

opt_wG = optim.Adam(wgan_G.parameters(), lr=1e-4, betas=(0.0, 0.9))
opt_wC = optim.Adam(wgan_C.parameters(), lr=1e-4, betas=(0.0, 0.9))

lambda_gp = 10
n_critic = 5
wgan_epochs = 10

wgan_G_losses = []
wgan_C_losses = []
wgan_fixed_noise = torch.randn(64, nz, 1, 1, device=device)
wgan_epoch_samples = {}

print("Starting WGAN-GP training...")
for epoch in range(wgan_epochs):
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        b_size = real_imgs.size(0)

        # --- Train Critic (n_critic steps) ---
        for _ in range(n_critic):
            noise = torch.randn(b_size, nz, 1, 1, device=device)
            fake_imgs = wgan_G(noise).detach()

            critic_real = wgan_C(real_imgs).mean()
            critic_fake = wgan_C(fake_imgs).mean()
            gp = compute_gradient_penalty(wgan_C, real_imgs, fake_imgs, device)

            loss_c = critic_fake - critic_real + lambda_gp * gp

            opt_wC.zero_grad()
            loss_c.backward()
            opt_wC.step()

        # --- Train Generator ---
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake_imgs = wgan_G(noise)
        loss_g = -wgan_C(fake_imgs).mean()

        opt_wG.zero_grad()
        loss_g.backward()
        opt_wG.step()

        wgan_G_losses.append(loss_g.item())
        wgan_C_losses.append(loss_c.item())

    with torch.no_grad():
        wgan_epoch_samples[epoch] = wgan_G(wgan_fixed_noise).cpu()

    print(
        f"Epoch [{epoch+1}/{wgan_epochs}]  "
        f"C_loss: {loss_c.item():.4f}  G_loss: {loss_g.item():.4f}"
    )

print("WGAN-GP training complete.")

### 6.4 Compare WGAN-GP vs Vanilla GAN Stability

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

window = 50

# Vanilla DCGAN losses
axes[0].plot(rolling_average(G_losses, window), label="G loss", linewidth=1.0)
axes[0].plot(rolling_average(D_losses, window), label="D loss", linewidth=1.0)
axes[0].set_title("Vanilla DCGAN")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Loss")
axes[0].legend()

# WGAN-GP losses
axes[1].plot(rolling_average(wgan_G_losses, window), label="G loss", linewidth=1.0)
axes[1].plot(rolling_average(wgan_C_losses, window), label="Critic loss", linewidth=1.0)
axes[1].set_title("WGAN-GP")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.suptitle("Training stability comparison", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side final samples
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

grid_dcgan = vutils.make_grid(
    epoch_samples[num_epochs - 1], nrow=8, normalize=True, padding=2
)
axes[0].imshow(grid_dcgan.permute(1, 2, 0).numpy(), cmap="gray")
axes[0].set_title("Vanilla DCGAN (final)")
axes[0].axis("off")

grid_wgan = vutils.make_grid(
    wgan_epoch_samples[wgan_epochs - 1], nrow=8, normalize=True, padding=2
)
axes[1].imshow(grid_wgan.permute(1, 2, 0).numpy(), cmap="gray")
axes[1].set_title("WGAN-GP (final)")
axes[1].axis("off")

plt.suptitle("Sample quality comparison", fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 7: Conditional GAN (cGAN)

A conditional GAN conditions both the generator and discriminator on additional
information -- in this case, the class label. This allows us to generate
specific digits on demand.

### 7.1 Conditional Generator and Discriminator

In [ ]:
n_classes = 10
embed_dim = 10


class ConditionalGenerator(nn.Module):
    """Generator conditioned on class label via embedding concatenation."""

    def __init__(self):
        super().__init__()
        self.label_embed = nn.Embedding(n_classes, embed_dim)
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz + embed_dim, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z, labels):
        label_emb = self.label_embed(labels).unsqueeze(2).unsqueeze(3)
        x = torch.cat([z, label_emb], dim=1)
        return self.main(x)


class ConditionalDiscriminator(nn.Module):
    """Discriminator conditioned on class label via channel-wise embedding."""

    def __init__(self):
        super().__init__()
        self.label_embed = nn.Embedding(n_classes, 32 * 32)
        self.main = nn.Sequential(
            # (nc+1 x 32 x 32) -> (ndf x 16 x 16)
            nn.Conv2d(nc + 1, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x, labels):
        label_map = self.label_embed(labels).view(-1, 1, 32, 32)
        x = torch.cat([x, label_map], dim=1)
        return self.main(x).view(-1)

### 7.2 Train Conditional GAN

In [ ]:
torch.manual_seed(42)

cgan_G = ConditionalGenerator().to(device)
cgan_D = ConditionalDiscriminator().to(device)
cgan_G.apply(weights_init)
cgan_D.apply(weights_init)

opt_cG = optim.Adam(cgan_G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_cD = optim.Adam(cgan_D.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion = nn.BCELoss()

cgan_epochs = 10
cgan_G_losses = []
cgan_D_losses = []

print("Starting cGAN training...")
for epoch in range(cgan_epochs):
    for i, (real_imgs, labels) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        labels = labels.to(device)
        b_size = real_imgs.size(0)

        label_real = torch.ones(b_size, device=device)
        label_fake = torch.zeros(b_size, device=device)

        # --- Train Discriminator ---
        cgan_D.zero_grad()
        output_real = cgan_D(real_imgs, labels)
        loss_d_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake_labels = torch.randint(0, n_classes, (b_size,), device=device)
        fake_imgs = cgan_G(noise, fake_labels)
        output_fake = cgan_D(fake_imgs.detach(), fake_labels)
        loss_d_fake = criterion(output_fake, label_fake)

        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        opt_cD.step()

        # --- Train Generator ---
        cgan_G.zero_grad()
        output_fake = cgan_D(fake_imgs, fake_labels)
        loss_g = criterion(output_fake, label_real)
        loss_g.backward()
        opt_cG.step()

        cgan_G_losses.append(loss_g.item())
        cgan_D_losses.append(loss_d.item())

    print(
        f"Epoch [{epoch+1}/{cgan_epochs}]  "
        f"D_loss: {loss_d.item():.4f}  G_loss: {loss_g.item():.4f}"
    )

print("cGAN training complete.")

### 7.3 Generate Specific Digits

In [ ]:
cgan_G.eval()

n_per_class = 8

fig, axes = plt.subplots(n_classes, 1, figsize=(n_per_class * 1.2, n_classes * 1.3))

with torch.no_grad():
    for digit in range(n_classes):
        noise = torch.randn(n_per_class, nz, 1, 1, device=device)
        labels = torch.full((n_per_class,), digit, dtype=torch.long, device=device)
        generated = cgan_G(noise, labels)
        grid = vutils.make_grid(generated, nrow=n_per_class, normalize=True, padding=1)
        axes[digit].imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap="gray")
        axes[digit].set_ylabel(str(digit), fontsize=14, rotation=0, labelpad=20)
        axes[digit].set_xticks([])
        axes[digit].set_yticks([])

cgan_G.train()
plt.suptitle("Conditional generation: each row is a specific digit", fontsize=14)
plt.tight_layout()
plt.show()

### 7.4 Conditional Latent Interpolation

Interpolate in latent space while holding the class label fixed.

In [ ]:
torch.manual_seed(42)
cgan_G.eval()

fig, axes = plt.subplots(n_classes, 1, figsize=(12, n_classes * 1.3))
n_steps = 10

with torch.no_grad():
    for digit in range(n_classes):
        z1 = torch.randn(1, nz, 1, 1, device=device)
        z2 = torch.randn(1, nz, 1, 1, device=device)
        z_interp = slerp(z1.squeeze(), z2.squeeze(), n_steps)
        z_interp = z_interp.view(n_steps, nz, 1, 1)
        labels = torch.full((n_steps,), digit, dtype=torch.long, device=device)
        imgs = cgan_G(z_interp, labels)
        grid = vutils.make_grid(imgs, nrow=n_steps, normalize=True, padding=1)
        axes[digit].imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap="gray")
        axes[digit].set_ylabel(str(digit), fontsize=14, rotation=0, labelpad=20)
        axes[digit].set_xticks([])
        axes[digit].set_yticks([])

cgan_G.train()
plt.suptitle("Conditional latent interpolation (fixed class, varying z)", fontsize=14)
plt.tight_layout()
plt.show()

---
## Summary

This notebook covered the core GAN variants from the ground up:

1. **Simple GAN on 2D data** -- demonstrated the minimax game between generator and discriminator with a toy distribution, making it easy to visualize how the generator learns to match the target.

2. **DCGAN** -- applied convolutional architectures with batch normalization, strided convolutions, and the DCGAN design guidelines to generate MNIST digits.

3. **Training dynamics** -- monitored D and G losses over time and introduced a simple ratio-based mode collapse indicator.

4. **Generated sample grids** -- tracked visual quality improvement across epochs.

5. **Latent space interpolation** -- verified smooth transitions using spherical interpolation (slerp), confirming the generator learned a continuous latent representation.

6. **WGAN-GP** -- replaced BCE loss with Wasserstein distance and gradient penalty for more stable training. Compared loss curves and sample quality against vanilla DCGAN.

7. **Conditional GAN** -- conditioned on class labels to enable targeted digit generation, with per-class interpolation.